# C8-embeddings — Practice p14 — Solution

In [ ]:
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["violin", "cello", "flute",
         "otter", "heron", "falcon",
         "bread", "cheese", "soup"]

V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))

C_raw = W.reshape(3, 3, 100).mean(axis=1)
centroid_norms = np.sqrt((C_raw * C_raw).sum(axis=1))
# Averaging distinct unit directions shortens every theme centroid below one.
C = C_raw / np.sqrt((C_raw * C_raw).sum(axis=1, keepdims=True))

A = C @ W.T
assign = np.argmax(A, axis=0)
assign_ok = bool(np.array_equal(assign, np.repeat(np.arange(3), 3)))

sorted_scores = np.sort(A, axis=0)
margins = sorted_scores[-1] - sorted_scores[-2]
hardest = WORDS[int(np.argmin(margins))]

All nine words return to their intended themes. `otter` has the smallest best-versus-second-best centroid gap, about 0.6746, so it is the least decisive assignment.

### Answer check

In [ ]:
assert W.shape == (9, 100) and W.dtype == np.float64
assert C_raw.shape == C.shape == (3, 100)
assert np.allclose(centroid_norms,
                   np.array([0.9514625490030788, 0.79755432656677, 0.8930255647784158]),
                   atol=1e-12, rtol=0)
assert np.all(centroid_norms < 1.0)
assert A.shape == (3, 9)
assert np.array_equal(assign, np.array([0, 0, 0, 1, 1, 1, 2, 2, 2]))
assert assign_ok is True
assert margins.shape == (9,)
assert np.isclose(margins.min(), 0.6746113334160365, atol=1e-12, rtol=0)
assert hardest == "otter"